# NYC Property Sales — Exploratory Analysis

This notebook analyzes NYC property sales from the cleaned dataset.

**Business question:**  
How has NYC property transaction activity changed from 2016–2025, and what property characteristics, locations, and transaction values are associated with those changes?

**Business Analyst framing:**  
NYC property-market stakeholders need to understand how transaction activity and the composition of properties being transacted have changed over time so they can identify emerging market segments, locations, and property characteristics that may warrant greater attention.

**Core analytical questions**
- How has NYC property transaction activity changed from 2016–2025?
- Has the composition of transactions changed over time?
- Has the size of properties being transacted changed over time?
- How have transaction values changed over time?
- Which boroughs and neighborhoods are experiencing the strongest transaction activity?
- Which boroughs and neighborhoods have the highest transaction values?
- Which property types are associated with higher transaction values and stronger activity?
- What property characteristics are associated with higher transaction values?

**Important methodology notes**
- Transaction activity includes all recorded ownership transfers, including `$0` sale-price records.
- Cash-based sale-price analysis uses only transactions with `SALE PRICE > 0`.
- Median sale price is used instead of average sale price because the dataset contains highly skewed transaction values.
- A minimum of 500 positive-price transactions is used for selected property-type and neighborhood comparisons. This threshold is for analysis only; it does not remove records from the cleaned dataset.
- Findings describe associations, not causal relationships.
- The dataset measures recorded property transactions/ownership transfers, not unique buyers or buyer preferences. Changes in transaction mix should not be interpreted as proof that buyers are choosing smaller properties or that fewer people are buying.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nyc_sales_property_cleaned.csv")

df["SALE DATE"] = pd.to_datetime(df["SALE DATE"], errors="coerce")
df["SALE YEAR"] = df["SALE DATE"].dt.year
df["PROPERTY AGE"] = df["SALE YEAR"] - df["YEAR BUILT"]

df["PRICE PER SQ FT"] = np.where(
    (df["GROSS SQUARE FEET"] > 0) & (df["SALE PRICE"] > 0),
    df["SALE PRICE"] / df["GROSS SQUARE FEET"],
    np.nan
)

borough_names = {
    1: "Manhattan",
    2: "Bronx",
    3: "Brooklyn",
    4: "Queens",
    5: "Staten Island"
}

df["BOROUGH NAME"] = df["BOROUGH"].map(borough_names)

print("Dataset shape:", df.shape)
print("Date range:", df["SALE YEAR"].min(), "to", df["SALE YEAR"].max())
print("Unique boroughs:", df["BOROUGH NAME"].nunique())

Dataset shape: (650335, 24)
Date range: 2016 to 2025
Unique boroughs: 5


## 1. Transaction Activity

First, measure overall sales activity by year, borough, and neighborhood.

These analyses use **all recorded transactions**, including `$0` sale-price records, because `$0` records represent ownership transfers and are relevant to transaction activity.

In [2]:
sales_by_year = (
    df.groupby("SALE YEAR")
    .size()
    .reset_index(name="TRANSACTION COUNT")
    .sort_values("SALE YEAR")
)

print("Transactions by year:")
print(sales_by_year.to_string(index=False))


sales_by_borough = (
    df.groupby("BOROUGH NAME")
    .size()
    .reset_index(name="TRANSACTION COUNT")
    .sort_values("TRANSACTION COUNT", ascending=False)
)

print("\nTransactions by borough:")
print(sales_by_borough.to_string(index=False))


sales_by_neighborhood = (
    df.groupby("NEIGHBORHOOD")
    .size()
    .reset_index(name="TRANSACTION COUNT")
    .sort_values("TRANSACTION COUNT", ascending=False)
)

print("\nTop 20 neighborhoods by transaction count:")
print(sales_by_neighborhood.head(20).to_string(index=False))

Transactions by year:
 SALE YEAR  TRANSACTION COUNT
      2016              80068
      2017              79674
      2018              76659
      2019              61222
      2020              51061
      2021              69256
      2022              65018
      2023              53318
      2024              54812
      2025              59247

Transactions by borough:
 BOROUGH NAME  TRANSACTION COUNT
       Queens             215624
     Brooklyn             196231
    Manhattan             104926
Staten Island              78168
        Bronx              55386

Top 20 neighborhoods by transaction count:
           NEIGHBORHOOD  TRANSACTION COUNT
         FLUSHING-NORTH              22500
     BEDFORD STUYVESANT              12348
           BOROUGH PARK              11808
                ASTORIA               9920
                BAYSIDE               9499
UPPER EAST SIDE (59-79)               8737
          EAST NEW YORK               8236
UPPER EAST SIDE (79-96)             

## 2. Transaction Composition Over Time

The overall transaction count does not show whether the kinds of properties being transacted have changed. This section tests whether the **composition of recorded transactions** has shifted over time.

Because the dataset records properties/transactions rather than unique buyers, changes in property mix should be interpreted as changes in the properties represented in the transaction records. It should not be interpretted not as direct evidence of changing buyer preferences.

In [3]:
property_mix = (
    df.groupby(["SALE YEAR", "BUILDING CLASS CATEGORY"])
    .size()
    .reset_index(name="TRANSACTION_COUNT")
)

property_mix["YEAR_TOTAL"] = property_mix.groupby("SALE YEAR")["TRANSACTION_COUNT"].transform("sum")
property_mix["TRANSACTION_SHARE_PCT"] = property_mix["TRANSACTION_COUNT"] / property_mix["YEAR_TOTAL"] * 100

top_property_types = df["BUILDING CLASS CATEGORY"].value_counts().head(10).index
property_mix_top = (
    property_mix[property_mix["BUILDING CLASS CATEGORY"].isin(top_property_types)]
    .sort_values(["SALE YEAR", "TRANSACTION_SHARE_PCT"], ascending=[True, False])
)

print("Transaction mix for the 10 most common property types:")
print(property_mix_top.to_string(index=False))

Transaction mix for the 10 most common property types:
 SALE YEAR           BUILDING CLASS CATEGORY  TRANSACTION_COUNT  YEAR_TOTAL  TRANSACTION_SHARE_PCT
      2016           01 ONE FAMILY DWELLINGS              18440       80068              23.030424
      2016           02 TWO FAMILY DWELLINGS              16339       80068              20.406405
      2016    10 COOPS - ELEVATOR APARTMENTS              13327       80068              16.644602
      2016   13 CONDOS - ELEVATOR APARTMENTS               9265       80068              11.571414
      2016         03 THREE FAMILY DWELLINGS               4505       80068               5.626468
      2016    07 RENTALS - WALKUP APARTMENTS               3754       80068               4.688515
      2016      09 COOPS - WALKUP APARTMENTS               2791       80068               3.485787
      2016             04 TAX CLASS 1 CONDOS               1175       80068               1.467503
      2016                22 STORE BUILDINGS          

### Property Size Composition Over Time

To test whether the size of transacted properties has shifted, compare the median gross square footage of positive-price transactions with a recorded gross square footage greater than zero.

In [4]:
size_by_year = (
    positive_sales[positive_sales["GROSS SQUARE FEET"] > 0]
    .groupby("SALE YEAR")
    .agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_GROSS_SQFT=("GROSS SQUARE FEET", "median"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
    .sort_values("SALE YEAR")
)

print("Median property size and median sale price by year:")
print(size_by_year.to_string(index=False))

NameError: name 'positive_sales' is not defined

## 3. Transaction Value Over Time

Transaction activity and transaction value are separate measures. This section evaluates whether cash-based transaction values changed over time, using only positive-price transactions.

In [ ]:
value_by_year = (
    positive_sales
    .groupby("SALE YEAR")
    .agg(
        POSITIVE_PRICE_TRANSACTIONS=("SALE PRICE", "count"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median"),
        TOTAL_POSITIVE_SALE_VALUE=("SALE PRICE", "sum")
    )
    .reset_index()
    .sort_values("SALE YEAR")
)

print("Cash-based transaction value by year:")
print(value_by_year.to_string(index=False))

Cash-based transaction value by year:
 SALE YEAR  POSITIVE_PRICE_TRANSACTIONS  MEDIAN_SALE_PRICE  TOTAL_POSITIVE_SALE_VALUE
      2016                        56670           570000.0                93173824841
      2017                        56822           625000.0                80149484278
      2018                        54478           650000.0                87214892780
      2019                        40071           745000.0                85835367907
      2020                        31031           750000.0                60881131164
      2021                        47053           860000.0                94355298656
      2022                        43460           910000.0               110856524796
      2023                        32939           895000.0                67204311481
      2024                        33686           900000.0                69297599732
      2025                        35451           950000.0                80858482937


## 4. Property Type Activity and Transaction Value

Property types are compared using both transaction volume and median sale price.

For value analysis, only positive-price transactions are included. Categories with fewer than 500 positive-price transactions are excluded from the ranked comparison to reduce the influence of very small samples.

In [ ]:
positive_sales = df[df["SALE PRICE"] > 0].copy()

sales_by_property_type = (
    df.groupby("BUILDING CLASS CATEGORY")
    .size()
    .reset_index(name="TRANSACTION COUNT")
    .sort_values("TRANSACTION COUNT", ascending=False)
)

print("Top 20 property types by transaction count:")
print(sales_by_property_type.head(20).to_string(index=False))


property_type_value = (
    positive_sales
    .groupby("BUILDING CLASS CATEGORY")
    .agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
)

property_type_value_filtered = (
    property_type_value[
        property_type_value["TRANSACTION_COUNT"] >= 500
    ]
    .sort_values("MEDIAN_SALE_PRICE", ascending=False)
)

print("\nProperty types with at least 500 positive-price transactions:")
print(property_type_value_filtered.to_string(index=False))

Top 20 property types by transaction count:
          BUILDING CLASS CATEGORY  TRANSACTION COUNT
          01 ONE FAMILY DWELLINGS             184881
          02 TWO FAMILY DWELLINGS             157242
  13 CONDOS - ELEVATOR APARTMENTS             101952
        03 THREE FAMILY DWELLINGS              44191
   10 COOPS - ELEVATOR APARTMENTS              40184
   07 RENTALS - WALKUP APARTMENTS              29885
            04 TAX CLASS 1 CONDOS              12490
15 CONDOS - 2-10 UNIT RESIDENTIAL              12071
               22 STORE BUILDINGS               8818
     09 COOPS - WALKUP APARTMENTS               8457
    12 CONDOS - WALKUP APARTMENTS               6707
           14 RENTALS - 4-10 UNIT               6138
                 44 CONDO PARKING               4899
 08 RENTALS - ELEVATOR APARTMENTS               4447
              21 OFFICE BUILDINGS               3444
                   17 CONDO COOPS               3098
                    30 WAREHOUSES               2905
  

In [ ]:
print("Extreme property age records:")
print(
    df.loc[
        df["PROPERTY AGE"] > 200,
        [
            "BOROUGH",
            "NEIGHBORHOOD",
            "BUILDING CLASS CATEGORY",
            "YEAR BUILT",
            "SALE YEAR",
            "PROPERTY AGE",
            "SALE PRICE"
        ]
    ].sort_values("PROPERTY AGE", ascending=False).head(20)
)

print("\nExtreme sale price records:")
print(
    df.loc[
        df["SALE PRICE"] > 100_000_000,
        [
            "BOROUGH",
            "NEIGHBORHOOD",
            "BUILDING CLASS CATEGORY",
            "YEAR BUILT",
            "SALE YEAR",
            "PROPERTY AGE",
            "SALE PRICE"
        ]
    ].sort_values("SALE PRICE", ascending=False).head(20)
)

Extreme property age records:
        BOROUGH        NEIGHBORHOOD            BUILDING CLASS CATEGORY  \
290846        5         PRINCES BAY            01 ONE FAMILY DWELLINGS   
610196        3          OCEAN HILL             14 RENTALS - 4-10 UNIT   
604021        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
615090        1        CIVIC CENTER  15 CONDOS - 2-10 UNIT RESIDENTIAL   
615091        1        CIVIC CENTER  15 CONDOS - 2-10 UNIT RESIDENTIAL   
604017        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
619733        1                SOHO             14 RENTALS - 4-10 UNIT   
604024        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604023        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604025        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604022        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604020        3  DOWNTOWN-METROTECH    13 CONDOS - ELEVATOR APARTMENTS   
604019  

## 5. Location and Transaction Value

Compare median sale prices across boroughs and neighborhoods.

The neighborhood comparison uses at least 500 positive-price transactions so that a small number of unusually expensive transactions does not dominate the ranking.

In [ ]:
borough_value = (
    positive_sales
    .groupby("BOROUGH NAME")["SALE PRICE"]
    .median()
    .reset_index(name="MEDIAN SALE PRICE")
    .sort_values("MEDIAN SALE PRICE", ascending=False)
)

print("Median sale price by borough:")
print(borough_value.to_string(index=False))

neighborhood_value = (
    positive_sales
    .groupby("NEIGHBORHOOD")
    .agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
)

neighborhood_value_filtered = (
    neighborhood_value[
        neighborhood_value["TRANSACTION_COUNT"] >= 500
    ]
    .sort_values("MEDIAN_SALE_PRICE", ascending=False)
)

print("\nTop 20 neighborhoods by median sale price:")
print(neighborhood_value_filtered.head(20).to_string(index=False))

Median sale price by borough:
 BOROUGH NAME  MEDIAN SALE PRICE
    Manhattan          1470356.0
     Brooklyn           930000.0
       Queens           695000.0
        Bronx           575000.0
Staten Island           565000.0

Top 20 neighborhoods by median sale price:
             NEIGHBORHOOD  TRANSACTION_COUNT  MEDIAN_SALE_PRICE
            JAVITS CENTER                545          4900000.0
             CIVIC CENTER               1543          3230000.0
                     SOHO               1976          2950000.0
   GREENWICH VILLAGE-WEST               2804          2800000.0
                  TRIBECA               3189          2672906.0
                  FASHION                730          2400000.0
          CARROLL GARDENS               1210          2373577.0
                 FLATIRON               1881          2360000.0
              COBBLE HILL                778          2135000.0
              MIDTOWN CBD               1316          1997000.0
         BROOKLYN HEIGHT

## 6. `$0` Sale-Price Records

NYC property sales data contains records with a sale price of `$0`. These records are retained in the cleaned dataset because they represent ownership transfers without cash consideration.

The analysis below measures how common these records are over time. This distinction is important because transaction activity and cash transaction value are not the same measure.

In [ ]:
size_value = positive_sales[
    positive_sales["GROSS SQUARE FEET"] > 0
].copy()

size_value["SIZE QUARTILE"] = pd.qcut(
    size_value["GROSS SQUARE FEET"],
    q=4,
    labels=["Smallest", "Lower-Mid", "Upper-Mid", "Largest"]
)

size_value_summary = (
    size_value
    .groupby("SIZE QUARTILE", observed=True)
    .agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_GROSS_SQFT=("GROSS SQUARE FEET", "median"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
)

print("Sale price by property-size quartile:")
print(size_value_summary.to_string(index=False))

Sale price by property-size quartile:
SIZE QUARTILE  TRANSACTION_COUNT  MEDIAN_GROSS_SQFT  MEDIAN_SALE_PRICE
     Smallest              10665              720.0           520000.0
    Lower-Mid              10613              960.0           535000.0
    Upper-Mid              10639             1512.0           615000.0
      Largest              10618             2741.0           965000.0


In [ ]:
unit_value = positive_sales[
    positive_sales["TOTAL UNITS"] > 0
].copy()

unit_value["UNIT GROUP"] = pd.cut(
    unit_value["TOTAL UNITS"],
    bins=[0, 1, 2, 5, 10, float("inf")],
    labels=["1 Unit", "2 Units", "3-5 Units", "6-10 Units", "11+ Units"]
)

unit_value_summary = (
    unit_value
    .groupby("UNIT GROUP", observed=True)
    .agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_UNITS=("TOTAL UNITS", "median"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
)

print("\nSale price by total-unit group:")
print(unit_value_summary.to_string(index=False))


Sale price by total-unit group:
UNIT GROUP  TRANSACTION_COUNT  MEDIAN_UNITS  MEDIAN_SALE_PRICE
    1 Unit             237828           1.0           745000.0
   2 Units              92111           2.0           800000.0
 3-5 Units              36812           3.0          1090000.0
6-10 Units               8225           7.0          1825000.0
 11+ Units               8633          29.0          4655000.0


In [ ]:
age_value = positive_sales[
    positive_sales["PROPERTY AGE"] >= 0
].copy()

age_value["AGE GROUP"] = pd.cut(
    age_value["PROPERTY AGE"],
    bins=[-1, 10, 25, 50, 100, float("inf")],
    labels=[
        "0-10 Years",
        "11-25 Years",
        "26-50 Years",
        "51-100 Years",
        "100+ Years"
    ]
)

age_value_summary = (
    age_value
    .groupby("AGE GROUP", observed=True)
    .agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_AGE=("PROPERTY AGE", "median"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
)

print("\nSale price by property-age group:")
print(age_value_summary.to_string(index=False))


Sale price by property-age group:
   AGE GROUP  TRANSACTION_COUNT  MEDIAN_AGE  MEDIAN_SALE_PRICE
  0-10 Years              55006         4.0          1018249.0
 11-25 Years              38120        16.0           790000.0
 26-50 Years              42742        36.0           640000.0
51-100 Years             216576        83.0           700000.0
  100+ Years              79217       111.0           969000.0


## 7. Location × Property Type

Location and property type should not be viewed independently. The same property category can have substantially different transaction values across boroughs.

Only combinations with at least 500 positive-price transactions are included in the comparison.

In [ ]:
location_property_value = (
    positive_sales
    .groupby(["BOROUGH NAME", "BUILDING CLASS CATEGORY"])
    .agg(
        TRANSACTION_COUNT=("SALE PRICE", "count"),
        MEDIAN_SALE_PRICE=("SALE PRICE", "median")
    )
    .reset_index()
)

location_property_value_filtered = (
    location_property_value[
        location_property_value["TRANSACTION_COUNT"] >= 500
    ]
    .sort_values("MEDIAN_SALE_PRICE", ascending=False)
)

print(
    location_property_value_filtered.head(25).to_string(index=False)
)

BOROUGH NAME           BUILDING CLASS CATEGORY  TRANSACTION_COUNT  MEDIAN_SALE_PRICE
   Manhattan               21 OFFICE BUILDINGS                798         18139500.0
    Brooklyn  08 RENTALS - ELEVATOR APARTMENTS                609         10353000.0
   Manhattan                22 STORE BUILDINGS                628          7500000.0
   Manhattan           01 ONE FAMILY DWELLINGS               1035          7000000.0
   Manhattan            14 RENTALS - 4-10 UNIT                516          5090000.0
   Manhattan    07 RENTALS - WALKUP APARTMENTS               3765          4500000.0
   Manhattan  08 RENTALS - ELEVATOR APARTMENTS               1412          4411924.0
   Manhattan           02 TWO FAMILY DWELLINGS                785          3800000.0
      Queens                     30 WAREHOUSES                599          3500000.0
    Brooklyn                     30 WAREHOUSES                694          3450000.0
   Manhattan 15 CONDOS - 2-10 UNIT RESIDENTIAL               1670

## 8. `$0` Sale-Price Records

NYC property sales data contains records with a sale price of `$0`. These records are retained in the cleaned dataset because they represent ownership transfers without cash consideration.

The analysis below measures how common these records are over time. This distinction is important because transaction activity and cash transaction value are not the same measure.

In [ ]:
zero_price_by_year = (
    df.groupby("SALE YEAR")
    .agg(
        TOTAL_TRANSACTIONS=("SALE PRICE", "count"),
        ZERO_PRICE_TRANSACTIONS=(
            "SALE PRICE",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

zero_price_by_year["ZERO_PRICE_PCT"] = (
    zero_price_by_year["ZERO_PRICE_TRANSACTIONS"]
    / zero_price_by_year["TOTAL_TRANSACTIONS"]
    * 100
)

print(zero_price_by_year.to_string(index=False))

 SALE YEAR  TOTAL_TRANSACTIONS  ZERO_PRICE_TRANSACTIONS  ZERO_PRICE_PCT
      2016               80068                    23398       29.222661
      2017               79674                    22852       28.681879
      2018               76659                    22181       28.934633
      2019               61222                    21151       34.548038
      2020               51061                    20030       39.227591
      2021               69256                    22203       32.059316
      2022               65018                    21558       33.156972
      2023               53318                    20379       38.221614
      2024               54812                    21126       38.542655
      2025               59247                    23796       40.164059


## Key Analytical Findings

Based on the analyses above:

1. **Transaction activity varies materially over time.** Activity was highest in 2016–2017, declined through 2020, rebounded in 2021, and partially recovered by 2025.

2. **Transaction composition should be evaluated separately from transaction volume.** The property-type mix can change even when total transaction activity moves in the same or opposite direction.

3. **Property size should be evaluated as a time-varying characteristic.** The year-by-year size analysis tests whether the properties represented in positive-price transactions have become smaller, larger, or remained broadly stable over the study period. Any conclusion should be based on observed median gross square footage, not assumptions about buyer preferences.

4. **Cash-based transaction value should be evaluated separately from overall activity.** Positive-price transaction counts and median sale price provide a clearer view of cash transactions than total transaction counts alone.

5. **Queens and Brooklyn have the strongest transaction volume**, while Manhattan has the highest median transaction value among the five boroughs.

6. **Property type matters.** High-volume residential categories dominate activity, while several commercial and specialized property categories show substantially higher median transaction values.

7. **Neighborhood value is concentrated.** The highest-value neighborhoods meeting the 500-transaction threshold are concentrated primarily in Manhattan, with several Brooklyn neighborhoods also appearing among the leaders.

8. **Larger properties are associated with higher median sale prices.**

9. **Properties with more total units are associated with substantially higher median sale prices.**

10. **Property age does not have a simple linear relationship with transaction value.** The newest and oldest age groups show higher median values than several middle-aged groups.

11. **Location and property type interact strongly.** Transaction values for the same property category can differ substantially by borough.

12. **`$0` transactions are a major analytical consideration.** Their share rises from about 29% of transactions in 2016 to about 40% in 2025, so cash-based price analysis must be separated from overall transaction activity.

13. **The dataset does not identify unique buyers or buyer intent.** Changes in transaction count, property size, or property mix should be interpreted as changes in recorded property transactions—not as direct evidence that the number of buyers is falling or that buyers prefer smaller properties.

## Business Implications

The revised analysis is designed to support decisions around **where transaction activity is concentrated, how the mix of properties being transacted is changing, and where cash-based transaction values are strongest**.

Potential business uses include:
- prioritizing neighborhoods or boroughs for deeper market review;
- identifying property categories with high activity versus high transaction value;
- monitoring shifts in the composition and size of transacted properties over time;
- separating overall ownership-transfer activity from cash-based market value when evaluating trends.

These findings should be treated as signals for further investigation rather than forecasts or causal explanations.

## Methodology and Limitations

- `$0` sale-price records are included when measuring transaction activity.

- `$0` sale-price records are excluded from median sale-price and other cash-value comparisons.

- Median sale price is preferred to the mean because sale prices are highly skewed.

- The 500 positive-price transaction threshold is used only for selected comparative analyses.

- Property size, unit count, age, location, property type, and changes over time are analyzed as descriptive associations rather than causal drivers.

- The transaction-composition and property-size-over-time analyses describe changes in recorded properties/transactions. They do not identify unique buyers or prove changes in buyer preferences.

- The dataset does not provide buyer demographics, financing terms, buyer intent, property condition, or other factors that could explain transaction decisions.

- Therefore, findings should be used to identify market patterns and potential opportunities, not to make causal claims about why properties sell for particular values or why transaction patterns change.